# Joins

## Introduction

Real databases are split across many tables. Joins are the mechanism for bringing data from multiple tables together in a single query. This notebook covers `INNER JOIN`, `LEFT JOIN`, table aliases, the `USING` shorthand, and the critical distinction between one-to-one, one-to-many, and many-to-many relationships.

## Objectives

You will be able to:

- Write `INNER JOIN` queries using both `ON` and `USING`
- Use table aliases to write concise multi-table queries
- Distinguish `INNER JOIN` from `LEFT JOIN` and know when each is appropriate
- Explain primary keys, foreign keys, and how they define join relationships
- Predict the number of rows returned by one-to-many and many-to-many joins

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('data/join_statements/data.sqlite')
cur = conn.cursor()

The CRM database schema — primary keys are marked with `*`:

![CRM schema](assets/join_statements/Database-Schema.png)

---

## INNER JOIN with ON

A `JOIN` (which defaults to `INNER JOIN`) returns only rows where the join condition matches in both tables. Rows that exist in only one table are excluded.

```sql
SELECT *
FROM orderdetails
JOIN products
ON orderdetails.productCode = products.productCode
LIMIT 5;
```

Use `tableName.columnName` when the same column name exists in both tables to avoid ambiguity.

In [ ]:
cur.execute("""
    SELECT *
    FROM orderdetails
    JOIN products
    ON orderdetails.productCode = products.productCode
    LIMIT 5;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## USING Clause

When the join column has the **same name** in both tables, `USING(column)` is a shorthand for `ON tableA.column = tableB.column`.

In [ ]:
# Equivalent to the ON query above
cur.execute("""
    SELECT *
    FROM orderdetails
    JOIN products USING(productCode)
    LIMIT 5;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## Table Aliases

Long table names make `ON` clauses verbose. Give tables short aliases by placing the alias directly after the table name (no `AS` needed).

```sql
FROM orderdetails o
JOIN products p
ON o.productCode = p.productCode
```

In [ ]:
# Join customers to their sales rep employee — foreign key (not same name)
cur.execute("""
    SELECT c.customerName, c.city,
           e.firstName || ' ' || e.lastName AS salesRep,
           e.jobTitle
    FROM customers c
    JOIN employees e
    ON c.salesRepEmployeeNumber = e.employeeNumber
    ORDER BY salesRep
    LIMIT 10;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

Note the alias `e.firstName || ' ' || e.lastName` — in SQLite, `||` is the string concatenation operator.

---

## LEFT JOIN

A `LEFT JOIN` returns **all rows from the left table**, plus matching rows from the right table. Where there is no match, the right table's columns are `NULL`.

Use `LEFT JOIN` when you want to keep rows from the left table even if they have no counterpart in the right table.

![join types Venn diagram](assets/join_statements/venn.png)

In [ ]:
# All products, including any that have never been ordered
cur.execute("""
    SELECT p.productCode, p.productName, od.orderNumber
    FROM products p
    LEFT JOIN orderdetails od USING(productCode);
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]

print(f"Total rows: {len(df)}")
print(f"Products with no orders: {df.orderNumber.isnull().sum()}")
df[df.orderNumber.isnull()]

---

## Primary Keys and Foreign Keys

- **Primary key** — uniquely identifies each row in a table (marked `*` in the schema diagram). Auto-incremented when declared as `INTEGER PRIMARY KEY`.
- **Foreign key** — a column in one table that references the primary key of another. It is NOT guaranteed to be unique in the referencing table.

When you join on a foreign key, the result can have more rows than either input table — this is the root of one-to-many and many-to-many behaviour.

---

## One-to-Many Joins

A one-to-many relationship exists when one row in table A corresponds to multiple rows in table B. Joining products to productlines: each productline maps to many products, so the result has one productline row *repeated* for each product.

In [ ]:
cur.execute("SELECT COUNT(*) FROM products;").fetchone()  # 110 products

In [ ]:
cur.execute("SELECT COUNT(*) FROM productlines;").fetchone()  # 7 product lines

In [ ]:
# One-to-many: 7 product lines × 110 products → still 110 rows (each line repeated)
cur.execute("""
    SELECT *
    FROM products
    JOIN productlines USING(productLine)
    LIMIT 5;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
print(f"Result rows: {len(df)}")
df.head()

---

## Many-to-Many Joins

A many-to-many join happens when multiple rows in table A can match multiple rows in table B. The result size is the **product** of the match counts — it can explode quickly.

Example: joining offices and customers on `state`. There are 2 MA offices and 9 MA customers → the join produces 18 MA rows.

In [ ]:
cur.execute("""
    SELECT *
    FROM offices
    JOIN customers USING(state);
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
print(f"Total rows: {len(df)}")
print(f"MA rows: {len(df[df.state == 'MA'])}")  # 2 offices × 9 customers

Many-to-many joins on large tables can return billions of rows. Always think about cardinality before executing a broad join.

---

## Practice

Use the CRM database to answer the following questions.

In [ ]:
pconn = sqlite3.connect('data/join_statements_lab/data.sqlite')
pc = pconn.cursor()

In [ ]:
# Display the first name and last name of all employees in the Boston office


In [ ]:
# Are there any offices with zero employees? (Hint: LEFT JOIN + GROUP BY)


In [ ]:
# All employees with their office city and state — include employees who have no office
# Order by first name then last name


In [ ]:
# Customer contacts (first + last name) with their order number, order date, and status


In [ ]:
# Customer contacts with payment amounts and payment dates
# Sort descending by payment amount


In [ ]:
# Level up: customer contacts with product names, quantities, and order dates
# This requires joining 4 tables: customers → orders → orderdetails → products
# Sort descending by order date


---

## Summary

In this notebook you learned how to:

- Write `INNER JOIN` queries with `ON tableA.col = tableB.col` and the shorthand `USING(col)`
- Use table aliases (e.g., `FROM customers c`) to write concise multi-table queries
- Use `LEFT JOIN` to retain all left-table rows even when there is no matching right-table row
- Distinguish primary keys (unique row identifier) from foreign keys (reference to another table)
- Recognise one-to-many and many-to-many relationships and anticipate their impact on result size

Next: [05 — Subqueries](05_subqueries.ipynb)